# 11. Piso regulatorio y cargo por potencia

Implementación del piso de la declaración y del cargo por potencia mensual según la normativa vigente (tarea 13 del backlog, HU-04). Las funciones se encuentran en `src/regulacion.py`.

In [1]:
import sys
sys.path.append("../src")

In [2]:
import pandas as pd
from pathlib import Path
from modelado import asignar_estaciones, calcular_maximos_estacionales
from regulacion import calcular_piso, asignar_piso, calcular_hrp, calcular_cargo, cargo_estacion

In [3]:
ruta_serie = Path("../data/processed/SERIE_TRABAJO_CTRELEUW.csv")
serie = pd.read_csv(ruta_serie, parse_dates=["MES"])
serie.info()

<class 'pandas.DataFrame'>
RangeIndex: 93 entries, 0 to 92
Data columns (total 10 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   MES                        93 non-null     datetime64[us]
 1   MAX_REAL_MW                93 non-null     float64       
 2   POTENCIA_DECLARADA_MW      93 non-null     float64       
 3   EDICION                    93 non-null     str           
 4   DESCARGA                   93 non-null     str           
 5   REVISADO                   93 non-null     bool          
 6   POSIBLE_COPIA_DECLARACION  93 non-null     bool          
 7   TEMPERATURA_MEDIA_C        93 non-null     float64       
 8   TEMPERATURA_MAX_C          93 non-null     float64       
 9   TEMPERATURA_MIN_C          93 non-null     float64       
dtypes: bool(2), datetime64[us](1), float64(5), str(2)
memory usage: 7.7 KB


In [4]:
serie = asignar_estaciones(serie)

## 1. Piso de cada estación

Según el art. 6 de la Res. SE 22/2026, la declaración no puede ser inferior al 90 % de la máxima demanda registrada en la misma estación del año anterior. Para el verano 2025 (noviembre 2025 a abril 2026) rige un porcentaje transitorio del 75 %. El piso se calcula solo si la estación anterior está completa. A diferencia de la evaluación, no se descartan las estaciones con posible copia, porque el piso es una regla de CAMMESA aplicada a su propio dato registrado.

In [5]:
maximos_estacionales = calcular_maximos_estacionales(serie)

In [6]:
maximos_estacionales = calcular_piso(maximos_estacionales)

In [7]:
maximos_estacionales[["ESTACION", "ANIO_ESTACION", "MAXIMO_MW", "COMPLETA", "MAXIMO_ANTERIOR_MW", "PORCENTAJE_PISO", "PISO_MW"]]


,ESTACION,ANIO_ESTACION,MAXIMO_MW,COMPLETA,MAXIMO_ANTERIOR_MW,PORCENTAJE_PISO,PISO_MW
1,INVIERNO,2019,45.579,True,NaN,0.90,NaN
3,INVIERNO,2020,42.805,True,45.579,0.90,41.021
5,INVIERNO,2021,42.352,True,42.805,0.90,38.524
7,INVIERNO,2022,44.156,True,42.352,0.90,38.117
9,INVIERNO,2023,42.448,True,44.156,0.90,39.740
11,INVIERNO,2024,41.960,True,42.448,0.90,38.203
13,INVIERNO,2025,44.436,True,41.960,0.90,37.764
15,INVIERNO,2026,44.004,False,44.436,0.90,39.992
0,VERANO,2018,45.960,False,NaN,0.90,NaN
2,VERANO,2019,42.187,True,NaN,0.90,NaN


## 2. Declaraciones y piso desde el régimen actual

Se muestran las estaciones desde noviembre 2025, cuando empezó a regir la Res. SE 400/2025, con lo declarado, lo registrado y el piso de cada mes.

In [8]:
recientes = ((serie["ESTACION"] == "VERANO") & (serie["ANIO_ESTACION"] == 2025)) | ((serie["ESTACION"] == "INVIERNO") & (serie["ANIO_ESTACION"] == 2026))
serie.loc[recientes, ["MES", "ESTACION", "ANIO_ESTACION", "POTENCIA_DECLARADA_MW", "MAX_REAL_MW"]]

,MES,ESTACION,ANIO_ESTACION,POTENCIA_DECLARADA_MW,MAX_REAL_MW
82,2025-11-01,VERANO,2025,36.081,38.628
83,2025-12-01,VERANO,2025,36.081,43.840
84,2026-01-01,VERANO,2025,36.081,44.244
85,2026-02-01,VERANO,2025,36.081,44.132
86,2026-03-01,VERANO,2025,36.081,43.560
87,2026-04-01,VERANO,2025,36.081,39.348
88,2026-05-01,INVIERNO,2026,44.756,41.484
89,2026-06-01,INVIERNO,2026,44.756,44.004
90,2026-07-01,INVIERNO,2026,44.756,43.836
91,2026-08-01,INVIERNO,2026,44.756,43.248


In [9]:
serie = asignar_piso(serie, maximos_estacionales)
serie.loc[recientes, ["MES", "ESTACION", "ANIO_ESTACION", "MAX_REAL_MW", "POTENCIA_DECLARADA_MW", "PISO_MW"]]

,MES,ESTACION,ANIO_ESTACION,MAX_REAL_MW,POTENCIA_DECLARADA_MW,PISO_MW
82,2025-11-01,VERANO,2025,38.628,36.081,35.892
83,2025-12-01,VERANO,2025,43.840,36.081,35.892
84,2026-01-01,VERANO,2025,44.244,36.081,35.892
85,2026-02-01,VERANO,2025,44.132,36.081,35.892
86,2026-03-01,VERANO,2025,43.560,36.081,35.892
87,2026-04-01,VERANO,2025,39.348,36.081,35.892
88,2026-05-01,INVIERNO,2026,41.484,44.756,39.992
89,2026-06-01,INVIERNO,2026,44.004,44.756,39.992
90,2026-07-01,INVIERNO,2026,43.836,44.756,39.992
91,2026-08-01,INVIERNO,2026,43.248,44.756,39.992


## 3. Cargo por potencia mensual

El cargo de cada mes sigue el punto 7.2 de la Res. SE 400/2025:

$$\text{cargo} = \text{precio} \times \text{KP} \times \max(\text{registrado}, \text{declarado}) \times \text{FPunta} \times \text{HRP}$$

- **Precio de la potencia:** 16 dólares por MWhrp.
- **KP:** 1,1 en diciembre, enero, febrero, junio, julio y agosto, y 0,9 en los demás meses (punto 6.5).
- **FPunta:** 1, el valor inicial de la norma.
- **HRP del mes:** horas de remuneración de la potencia. Se calculan como los días de lunes a viernes por 18 horas fuera del valle, a partir de la semana típica de 90 HRP del punto 6.5. Los feriados no se descuentan.

In [10]:
serie = calcular_hrp(serie)

In [11]:
serie[["MES", "DIAS_HABILES", "HRP"]].head()

,MES,DIAS_HABILES,HRP
0,2019-01-01,23,414
1,2019-02-01,20,360
2,2019-03-01,21,378
3,2019-04-01,22,396
4,2019-05-01,23,414


In [12]:
serie = calcular_cargo(serie)

In [13]:
recientes = ((serie["ESTACION"] == "VERANO") & (serie["ANIO_ESTACION"] == 2025)) | ((serie["ESTACION"] == "INVIERNO") & (serie["ANIO_ESTACION"] == 2026))
serie.loc[recientes, ["MES", "MAX_REAL_MW", "POTENCIA_DECLARADA_MW", "REQ_POT_MW", "KP", "HRP", "CARGO_USD"]]

,MES,MAX_REAL_MW,POTENCIA_DECLARADA_MW,REQ_POT_MW,KP,HRP,CARGO_USD
82,2025-11-01,38.628,36.081,38.628,0.9,360,200247.55
83,2025-12-01,43.840,36.081,43.840,1.1,414,319435.78
84,2026-01-01,44.244,36.081,44.244,1.1,396,308362.98
85,2026-02-01,44.132,36.081,44.132,1.1,360,279620.35
86,2026-03-01,43.560,36.081,43.560,0.9,396,248396.54
87,2026-04-01,39.348,36.081,39.348,0.9,396,224378.04
88,2026-05-01,41.484,44.756,44.756,0.9,378,243615.86
89,2026-06-01,44.004,44.756,44.756,1.1,396,311931.42
90,2026-07-01,43.836,44.756,44.756,1.1,414,326110.12
91,2026-08-01,43.248,44.756,44.756,1.1,378,297752.72


## 4. Cargo por estación

El cargo de las estaciones anteriores a noviembre 2025 se calcula con las reglas actuales, aunque entonces regían otras. Es una referencia contrafactual, no lo que la Cooperativa pagó.

In [14]:
cargo_estacional = cargo_estacion(serie)
cargo_estacional

ESTACION  ANIO_ESTACION
INVIERNO  2019             1684893.74
          2020             1540243.04
          2021             1539452.51
          2022             1576942.53
          2023             1560198.24
          2024             1503904.32
          2025             1561211.80
          2026             1434626.73
VERANO    2018             1101936.42
          2019             1518004.17
          2020             1541275.05
          2021             1571411.63
          2022             1592856.35
          2023             1531327.97
          2024             1532244.20
          2025             1580441.24
Name: CARGO_USD, dtype: float64

## Conclusión

- El piso del verano 2025 es de 35,89 MW (75 % de 47,86 MW) y el del invierno 2026, de 39,99 MW (90 % de 44,44 MW).
- En el verano 2025 se declararon 36,08 MW, apenas por encima del piso, y la demanda registrada superó lo declarado en los seis meses. El cargo se calculó sobre lo registrado, así que la declaración no lo afectó.
- En el invierno 2026 se declararon 44,76 MW, por encima de la demanda registrada en todos los meses con datos. El cargo se calculó sobre lo declarado, esa diferencia es el sobrecosto evitable que se cuantifica en la evaluación.